# <font color = 'red'> Dependencias

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, create_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles)
from visualization_tools import plot_interactive_chart
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> Carga de Datos

In [2]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [3]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
No duplicate rows found.


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 85 columns):
 #   Column                                              Non-Null Count   Dtype  
---  ------                                              --------------   -----  
 0   Customer_ID                                         100000 non-null  object 
 1   Age                                                 100000 non-null  int64  
 2   Annual_Income                                       100000 non-null  float64
 3   Monthly_Inhand_Salary                               100000 non-null  float64
 4   Num_Bank_Accounts                                   100000 non-null  int64  
 5   Num_Credit_Card                                     100000 non-null  int64  
 6   Interest_Rate                                       100000 non-null  float64
 7   Num_of_Loan                                         100000 non-null  int64  
 8   Delay_from_due_date                                 100000 non-nu

# <font color = 'red'> Análisis

In [5]:
# Mediana de la edad para los deudores malos: 30 años.
# Mediana de la edad para los deudores standard: 32 años.
# Mediana de la edad para los deudores buenos: 37 años.

# La median de la edad para los deudores malos y standard pareciera estadísticamente no significativa:

fig_box = px.box(df, x="Credit_Mix", y="Age", title="Distribution of Age by Credit Score Category")
fig_box.show()

In [6]:
# Análisis por deciles
analysis_summary = summarize_decile_analysis(df, "Age", target_col="Credit_Mix")

deciles = analysis_summary['decile_summary']
df_deciles = analysis_summary['df_deciles']

deciles

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_count_Bad,prop_count_Good,prop_count_Standard
Decile,,,,,,,,,,
0,14,19,11188,0.11188,4275,1435,5478,0.382106,0.128262,0.489632
1,20,23,11238,0.11238,2809,3176,5253,0.249956,0.282613,0.467432
2,24,26,8769,0.08769,2330,2457,3982,0.265709,0.280192,0.454100
3,27,29,8815,0.08815,2274,2332,4209,0.257969,0.264549,0.477482
4,30,33,11438,0.11438,3050,3223,5165,0.266655,0.281780,0.451565
5,34,36,8816,0.08816,2245,2440,4131,0.254651,0.276770,0.468580
6,37,40,11519,0.11519,3127,3028,5364,0.271465,0.262870,0.465665
7,41,43,8322,0.08322,1995,2243,4084,0.239726,0.269527,0.490747
8,44,48,9955,0.09955,1663,3974,4318,0.167052,0.399196,0.433752


In [33]:
df_deciles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 86 columns):
 #   Column                                              Non-Null Count   Dtype  
---  ------                                              --------------   -----  
 0   Customer_ID                                         100000 non-null  object 
 1   Age                                                 100000 non-null  int64  
 2   Annual_Income                                       100000 non-null  float64
 3   Monthly_Inhand_Salary                               100000 non-null  float64
 4   Num_Bank_Accounts                                   100000 non-null  int64  
 5   Num_Credit_Card                                     100000 non-null  int64  
 6   Interest_Rate                                       100000 non-null  float64
 7   Num_of_Loan                                         100000 non-null  int64  
 8   Delay_from_due_date                                 100000 non-nu

In [7]:
df[(df['Age'] >= 37) & (df['Age'] <= 40) & (df['Credit_Score'] == 2)].shape

(3028, 85)

<font color = 'skyblue'> Good

In [8]:
# La relación entre la edad y el impago inversa: a medida que los deudores tienen mayor edad mayor la proporción de buenos, es decir, menor
# la proporción de deudores standard o malos.
# Sin embargo, entre los deciles 1 y 7 (20 a 48 años) la proporción de buenos es muy similar por lo que se puede hacer agrupaciones:
#   Menores de 20 años.
#   Entre 20 y 43 años.
#   Mayores a 43 años.

chart_types = {
    "prop_count_Good": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles,  
    y_columns=["prop_count_Good", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title="Proportion of Credit Score Categories by Age Decile",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_count_Good"],  
    width=900,
    height=500,
    custom_colors={"prop_count_Good": "lightgreen", "Decile_Count": "gray"}  
)

fig.show()


In [9]:
deciles

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_count_Bad,prop_count_Good,prop_count_Standard
Decile,,,,,,,,,,
0,14,19,11188,0.11188,4275,1435,5478,0.382106,0.128262,0.489632
1,20,23,11238,0.11238,2809,3176,5253,0.249956,0.282613,0.467432
2,24,26,8769,0.08769,2330,2457,3982,0.265709,0.280192,0.454100
3,27,29,8815,0.08815,2274,2332,4209,0.257969,0.264549,0.477482
4,30,33,11438,0.11438,3050,3223,5165,0.266655,0.281780,0.451565
5,34,36,8816,0.08816,2245,2440,4131,0.254651,0.276770,0.468580
6,37,40,11519,0.11519,3127,3028,5364,0.271465,0.262870,0.465665
7,41,43,8322,0.08322,1995,2243,4084,0.239726,0.269527,0.490747
8,44,48,9955,0.09955,1663,3974,4318,0.167052,0.399196,0.433752


<font color = 'skyblue'> Standard

In [10]:
# La relación entre la edad y la proporción de standard no es tan clara, aunque a edades mayores
# (+48) la proporción de standard decrece.

chart_types = {
    "prop_count_Standard": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles,  
    y_columns=["prop_count_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title="Proportion of Credit Score Categories by Age Decile",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_count_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_count_Standard": "brown", "Decile_Count": "gray"}  
)

fig.show()

<font color = 'skyblue'> Bad

In [11]:
# A medida que los deudores son mayores menor es la proporción de malos clientes.
# 1) Menores a 20
# 2) Entre 20 y 43
# 3) Mayores a 43
chart_types = {
    "prop_count_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles,  
    y_columns=["prop_count_Bad", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title="Proportion of Credit Score Categories by Age Decile",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_count_Bad"],  
    width=900,
    height=500,
    custom_colors={"prop_count_Bad": "red", "Decile_Count": "gray"}  
)

fig.show()

In [12]:
deciles

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_count_Bad,prop_count_Good,prop_count_Standard
Decile,,,,,,,,,,
0,14,19,11188,0.11188,4275,1435,5478,0.382106,0.128262,0.489632
1,20,23,11238,0.11238,2809,3176,5253,0.249956,0.282613,0.467432
2,24,26,8769,0.08769,2330,2457,3982,0.265709,0.280192,0.454100
3,27,29,8815,0.08815,2274,2332,4209,0.257969,0.264549,0.477482
4,30,33,11438,0.11438,3050,3223,5165,0.266655,0.281780,0.451565
5,34,36,8816,0.08816,2245,2440,4131,0.254651,0.276770,0.468580
6,37,40,11519,0.11519,3127,3028,5364,0.271465,0.262870,0.465665
7,41,43,8322,0.08322,1995,2243,4084,0.239726,0.269527,0.490747
8,44,48,9955,0.09955,1663,3974,4318,0.167052,0.399196,0.433752


<font color = 'skyblue'> Todos

In [13]:
chart_types = {
    "prop_count_Good":"line",
    "prop_count_Standard":"line",
    "prop_count_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles,  
    y_columns=["prop_count_Bad", "prop_count_Good", "prop_count_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title="Proportion of Credit Score Categories by Age Decile",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_count_Bad", "prop_count_Good", "prop_count_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_count_Bad": "red", "prop_count_Good":"lightgreen",
    "prop_count_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

<font color = 'skyblue'> Regresiones

<font color = 'gold'> Sin Agrupaciones

In [14]:
# Como Credit_Score tiene tres categorías (Bad, Standard, Good) y hay un orden entre estas podemos usar dos enfoques 
# de regresión categórica:

# Regresión Logística Multinomial → No asume orden en las categorías (como si fueran colores: rojo, azul, verde).
# Regresión Logística Ordinal → Asume que hay un orden en las categorías (Bad < Standard < Good).
# Dado que hay un orden entre las categorías se utiliza Regresión Logística Ordinal:

In [15]:
# Ajustar modelo de regresión logística ordinal
model_ordinal = OrderedModel(df["Credit_Score"], df["Age"], distr="logit").fit()

# Mostrar resumen del modelo
print(model_ordinal.summary())

odds_ratio_age = np.exp(model_ordinal.params['Age'])
print(f"Odds Ratio de Age: {odds_ratio_age:.2f}",
      "Por cada año adicional de edad, la probabilidad de estar en una categoría superior (Standard o Good) aumenta en 4.4%.")

# Todos los coeficientes son sigificativos. La relación entre la edad y Credit_Score no es aleatoria.

# Age	0.0429	Por cada año adicional de edad, la probabilidad de estar en una categoría superior de Credit_Score aumenta.
# Threshold 0/1	0.2073	Umbral que separa las categorías Bad y Standard: un cliente con una "puntuación de regresión" 
# superior a 0.2073 es más probable que sea Standard en lugar de Bad.

# Threshold 1/2	0.7375	Umbral que separa las categorías Standard y Good. Si la puntuación supera 0.7375, es más probable que 
# sea Good en lugar de Standard.

Optimization terminated successfully.
         Current function value: 1.031093
         Iterations: 75
         Function evaluations: 135
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0311e+05
Model:                   OrderedModel   AIC:                         2.062e+05
Method:            Maximum Likelihood   BIC:                         2.063e+05
Date:                Sun, 02 Mar 2025                                         
Time:                        15:08:39                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------

<font color = 'gold'> Con Agrupaciones

In [16]:
group_map = {
    0: "Group_1", 
    1: "Group_2", 
    2: "Group_2",
    3: "Group_2", 
    4: "Group_2", 
    5: "Group_2",
    6: "Group_2", 
    7: "Group_2",
    8: "Group_3", 
    9: "Group_4"
}

df_grouped = group_deciles(df_deciles, group_map)

grouped_summary = summarize_grouped_deciles(df_grouped, continuous_variable="Age", target_col="Credit_Mix")

display(grouped_summary)


,Grouped_Decile,Grouped_Min,Grouped_Max,Grouped_Count,Grouped_Proportion,count_Bad,count_Good,count_Standard,prop_count_Bad,prop_count_Good,prop_count_Standard
0,Group_1,14,19,11188,0.11188,4275,1435,5478,0.382106,0.128262,0.489632
1,Group_2,20,43,68917,0.68917,17830,18899,32188,0.258717,0.274228,0.467055
2,Group_3,44,48,9955,0.09955,1663,3974,4318,0.167052,0.399196,0.433752
3,Group_4,49,56,9940,0.09940,0,6076,3864,0.000000,0.611268,0.388732


In [ ]:
chart_types = {
    "prop_count_Good": "line",   
    "Grouped_Count": "bar"    
}

fig = plot_interactive_chart(
    df=grouped_summary,  
    y_columns=["prop_count_Good", "Grouped_Count"],  
    x_column="Grouped_Decile",  
    chart_types=chart_types, 
    title="Proportion of Credit Score Categories by Age Grouped Decile",
    x_title="Grouped Decile",
    y_title="Grouped Count",  
    y2_title="Proportion",   
    secondary_y=["prop_count_Good"],  
    width=900,
    height=500,
    custom_colors={"prop_count_Good": "lightgreen", "Grouped_Count": "gray"}  
)

fig.show()


In [18]:
chart_types = {
    "prop_count_Bad": "line",  
    "Grouped_Count": "bar"    
}

fig = plot_interactive_chart(
    df=grouped_summary,  
    y_columns=["prop_count_Bad", "Grouped_Count"],  
    x_column="Grouped_Decile",  
    chart_types=chart_types, 
    title="Proportion of Credit Score Categories by Age Grouped Decile",
    x_title="Grouped Decile",
    y_title="Grouped Count",  
    y2_title="Proportion",   
    secondary_y=["prop_count_Bad"],  
    width=900,
    height=500,
    custom_colors={"prop_count_Bad": "red", "Grouped_Count": "gray"}  
)

fig.show()

In [19]:
chart_types = {
    "prop_count_Standard": "line",  
    "Grouped_Count": "bar"    
}

fig = plot_interactive_chart(
    df=grouped_summary,  
    y_columns=["prop_count_Standard", "Grouped_Count"],  
    x_column="Grouped_Decile",  
    chart_types=chart_types, 
    title="Proportion of Credit Score Categories by Age Grouped Decile",
    x_title="Grouped Decile",
    y_title="Grouped Count",  
    y2_title="Proportion",   
    secondary_y=["prop_count_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_count_Standard": "brown", "Grouped_Count": "gray"}  
)

fig.show()

In [23]:
chart_types = {
    "prop_count_Good": "line", 
    "prop_count_Standard": "line",  
    "prop_count_Bad": "line",
    "Grouped_Count": "bar"    
}

fig = plot_interactive_chart(
    df=grouped_summary,  
    y_columns=["prop_count_Good", "prop_count_Standard", "prop_count_Bad", "Grouped_Count"],  
    x_column="Grouped_Decile",  
    chart_types=chart_types, 
    title="Proportion of Credit Score Categories by Age Grouped Decile",
    x_title="Grouped Decile",
    y_title="Grouped Count",  
    y2_title="Proportion",   
    secondary_y=["prop_count_Good", "prop_count_Standard", "prop_count_Bad"],  
    width=900,
    height=500,
    custom_colors={"prop_count_Good": "lightgreen", 
                   "prop_count_Standard": "brown",
                   "prop_count_Bad": "red",
                   "Grouped_Count": "gray"}  
)

fig.show()